# Unstructured OCR → Maternal Registry LLM Demo

This notebook demonstrates **only the LLM extraction layer** for noisy OCR.

The OCR result may be:
- incomplete,
- unordered,
- poorly formatted,
- missing labels,
- full of abbreviations,
- split across unrelated lines,
- or contain OCR mistakes such as `O/0` and `I/1`.

The LLM therefore performs **document-level schema alignment**, not line-by-line parsing.

```text
Noisy / unordered OCR text
        ↓
Local LLM
        ├─ search entire text for evidence
        ├─ map evidence to registry fields
        ├─ normalize values / codes
        ├─ identify missing fields
        ├─ identify ambiguity
        └─ generate clarification questions
        ↓
Deterministic validation
        ↓
User review only where needed
        ↓
Final 31-column registry row
```

No model is installed or called here. The model output is mocked.


## 1. Target registry schema

In [1]:
TARGET_COLUMNS = ['id', 'age (years)', 'education level (0=none/primary,1=secondary,2=higher)', 'consanguinity', 'desired pregnancy', 'hypertension history', 'diabetes mellitus', 'gravidity (number)', 'parity (number)', 'abortions (number)', 'living children (number)', 'previous cesarean', 'bmi pregestational (kg/m2)', 'mean systolic bp (mmhg)', 'mean diastolic bp (mmhg)', 'hemoglobin (g/dl)', 'first fasting glucose (mg/dl)', 'proteinuria', 'hiv test result', 'syphilis test result', 'hepatitis c test result', 'gestational age at enrollment (weeks)', 'gestational dm', 'gestational age at birth (weeks)', 'preterm birth', 'type of delivery (0=vaginal,1=cesarean)', 'newborn sex (0=female,1=male)', 'child birth weight (g)', 'head circumference (cm)', 'breastfeeding initiated', 'referral to higher care']

print(f"{len(TARGET_COLUMNS)} target columns")
for i, col in enumerate(TARGET_COLUMNS, 1):
    print(f"{i:>2}. {col}")


31 target columns
 1. id
 2. age (years)
 3. education level (0=none/primary,1=secondary,2=higher)
 4. consanguinity
 5. desired pregnancy
 6. hypertension history
 7. diabetes mellitus
 8. gravidity (number)
 9. parity (number)
10. abortions (number)
11. living children (number)
12. previous cesarean
13. bmi pregestational (kg/m2)
14. mean systolic bp (mmhg)
15. mean diastolic bp (mmhg)
16. hemoglobin (g/dl)
17. first fasting glucose (mg/dl)
18. proteinuria
19. hiv test result
20. syphilis test result
21. hepatitis c test result
22. gestational age at enrollment (weeks)
23. gestational dm
24. gestational age at birth (weeks)
25. preterm birth
26. type of delivery (0=vaginal,1=cesarean)
27. newborn sex (0=female,1=male)
28. child birth weight (g)
29. head circumference (cm)
30. breastfeeding initiated
31. referral to higher care


## 2. Example of realistic noisy OCR

The fields are deliberately out of order, some are missing, and some values are only partially labeled.


In [2]:
ocr_text = '''
pt 201

28
sec

g2 p1 a0
1 living
no cs

wanted preg yes

24.8

bp maybe 12O 8O
hb12.4
fast 86

hiv neg
hcv neg

32 sem

notes:
follow up next month
'''.strip()

print(ocr_text)


pt 201

28
sec

g2 p1 a0
1 living
no cs

wanted preg yes

24.8

bp maybe 12O 8O
hb12.4
fast 86

hiv neg
hcv neg

32 sem

notes:
follow up next month


## 3. Prompt for document-level extraction

The critical instruction is that the model must **not assume order** and must **not invent missing values**.


In [3]:
import json

SYSTEM_PROMPT = """
You convert noisy OCR text from a maternal registry form into structured data.

The OCR text may be incomplete, unordered, noisy, and poorly formatted.

Do NOT assume that:
- fields appear in schema order,
- each field has a label,
- every field is present,
- one line corresponds to one field.

Read the OCR result as one unstructured document.

For every field in the target schema:
1. Search the entire OCR text for evidence.
2. Extract a value only when the evidence is sufficiently clear.
3. Normalize common abbreviations and formatting.
4. Map categorical values to the registry coding supplied below.
5. Never infer a missing value from medical knowledge or another field.
6. If no evidence exists, return NOT_PROVIDED.
7. If relevant text exists but cannot be interpreted, return ILLEGIBLE.
8. If more than one interpretation is plausible, return NEEDS_REVIEW.
9. For NEEDS_REVIEW, provide the candidate value if one exists and one concise question.
10. Include the exact OCR fragment used as evidence.
11. Return every target field, even if the value is null.
12. Return JSON only.
""".strip()

CODING_RULES = {
    "education": {"none/primary": 0, "secondary": 1, "higher": 2},
    "binary_no_yes": {"no": 0, "yes": 1},
    "test_result": {"negative": 0, "positive": 1},
    "delivery": {"vaginal": 0, "cesarean": 1},
    "newborn_sex": {"female": 0, "male": 1},
}

prompt_payload = {
    "ocr_text": ocr_text,
    "target_columns": TARGET_COLUMNS,
    "coding_rules": CODING_RULES,
    "status_values": ["KNOWN", "NEEDS_REVIEW", "NOT_PROVIDED", "ILLEGIBLE"],
    "temporary_output_fields": ["value", "status", "evidence", "reason", "question"]
}

print(SYSTEM_PROMPT)
print("\n--- INPUT TO LOCAL LLM ---\n")
print(json.dumps(prompt_payload, indent=2))


You convert noisy OCR text from a maternal registry form into structured data.

The OCR text may be incomplete, unordered, noisy, and poorly formatted.

Do NOT assume that:
- fields appear in schema order,
- each field has a label,
- every field is present,
- one line corresponds to one field.

Read the OCR result as one unstructured document.

For every field in the target schema:
1. Search the entire OCR text for evidence.
2. Extract a value only when the evidence is sufficiently clear.
3. Normalize common abbreviations and formatting.
4. Map categorical values to the registry coding supplied below.
5. Never infer a missing value from medical knowledge or another field.
6. If no evidence exists, return NOT_PROVIDED.
7. If relevant text exists but cannot be interpreted, return ILLEGIBLE.
8. If more than one interpretation is plausible, return NEEDS_REVIEW.
9. For NEEDS_REVIEW, provide the candidate value if one exists and one concise question.
10. Include the exact OCR fragment used a

## 4. Mock LLM output

The temporary LLM output is richer than the final CSV row.

Each field keeps:
- `value`
- `status`
- `evidence`
- `reason`
- `question`

The `evidence` field is especially useful for debugging noisy OCR.


In [4]:
def field(value=None, status="NOT_PROVIDED", evidence=None, reason=None, question=None):
    return {
        "value": value,
        "status": status,
        "evidence": evidence,
        "reason": reason,
        "question": question
    }

mock_llm_response = {
    "id": field(201, "KNOWN", "pt 201"),
    "age (years)": field(28, "KNOWN", "28"),
    "education level (0=none/primary,1=secondary,2=higher)": field(
        1, "KNOWN", "sec", "Abbreviation mapped to secondary."
    ),
    "consanguinity": field(),
    "desired pregnancy": field(1, "KNOWN", "wanted preg yes"),
    "hypertension history": field(),
    "diabetes mellitus": field(),
    "gravidity (number)": field(2, "KNOWN", "g2"),
    "parity (number)": field(1, "KNOWN", "p1"),
    "abortions (number)": field(0, "KNOWN", "a0"),
    "living children (number)": field(1, "KNOWN", "1 living"),
    "previous cesarean": field(0, "KNOWN", "no cs"),
    "bmi pregestational (kg/m2)": field(
        24.8,
        "NEEDS_REVIEW",
        "24.8",
        "The value is plausible for BMI, but the OCR fragment has no explicit BMI label.",
        "I found an unlabeled value 24.8. Is this the pregestational BMI?"
    ),
    "mean systolic bp (mmhg)": field(
        120,
        "NEEDS_REVIEW",
        "bp maybe 12O 8O",
        "OCR contains ambiguous O/0 characters.",
        "I read the blood pressure as 120/80. Is that correct?"
    ),
    "mean diastolic bp (mmhg)": field(
        80,
        "NEEDS_REVIEW",
        "bp maybe 12O 8O",
        "OCR contains ambiguous O/0 characters.",
        "I read the blood pressure as 120/80. Is that correct?"
    ),
    "hemoglobin (g/dl)": field(12.4, "KNOWN", "hb12.4"),
    "first fasting glucose (mg/dl)": field(86.0, "KNOWN", "fast 86"),
    "proteinuria": field(),
    "hiv test result": field(0, "KNOWN", "hiv neg"),
    "syphilis test result": field(),
    "hepatitis c test result": field(0, "KNOWN", "hcv neg"),
    "gestational age at enrollment (weeks)": field(
        32.0,
        "NEEDS_REVIEW",
        "32 sem",
        "The OCR indicates 32 weeks, but does not explicitly distinguish enrollment age from another gestational-age field.",
        "Does '32 sem' refer to gestational age at enrollment?"
    ),
    "gestational dm": field(),
    "gestational age at birth (weeks)": field(),
    "preterm birth": field(),
    "type of delivery (0=vaginal,1=cesarean)": field(),
    "newborn sex (0=female,1=male)": field(),
    "child birth weight (g)": field(),
    "head circumference (cm)": field(),
    "breastfeeding initiated": field(),
    "referral to higher care": field()
}

print(json.dumps(mock_llm_response, indent=2))


{
  "id": {
    "value": 201,
    "status": "KNOWN",
    "evidence": "pt 201",
    "reason": null,
    "question": null
  },
  "age (years)": {
    "value": 28,
    "status": "KNOWN",
    "evidence": "28",
    "reason": null,
    "question": null
  },
  "education level (0=none/primary,1=secondary,2=higher)": {
    "value": 1,
    "status": "KNOWN",
    "evidence": "sec",
    "reason": "Abbreviation mapped to secondary.",
    "question": null
  },
  "consanguinity": {
    "value": null,
    "status": "NOT_PROVIDED",
    "evidence": null,
    "reason": null,
    "question": null
  },
  "desired pregnancy": {
    "value": 1,
    "status": "KNOWN",
    "evidence": "wanted preg yes",
    "reason": null,
    "question": null
  },
  "hypertension history": {
    "value": null,
    "status": "NOT_PROVIDED",
    "evidence": null,
    "reason": null,
    "question": null
  },
  "diabetes mellitus": {
    "value": null,
    "status": "NOT_PROVIDED",
    "evidence": null,
    "reason": null,
    

## 5. Validate the temporary extraction object

Validation happens outside the LLM.

This catches:
- missing schema keys,
- unexpected fields,
- invalid status values,
- invalid categorical codes.


In [5]:
ALLOWED_STATUS = {"KNOWN", "NEEDS_REVIEW", "NOT_PROVIDED", "ILLEGIBLE"}

assert set(mock_llm_response) == set(TARGET_COLUMNS), "Schema mismatch"

for col, item in mock_llm_response.items():
    assert item["status"] in ALLOWED_STATUS, f"Invalid status in {col}"

coded_domains = {
    "education level (0=none/primary,1=secondary,2=higher)": {0, 1, 2},
    "consanguinity": {0, 1},
    "desired pregnancy": {0, 1},
    "hypertension history": {0, 1},
    "diabetes mellitus": {0, 1},
    "previous cesarean": {0, 1},
    "proteinuria": {0, 1},
    "hiv test result": {0, 1},
    "syphilis test result": {0, 1},
    "hepatitis c test result": {0, 1},
    "gestational dm": {0, 1},
    "preterm birth": {0, 1},
    "type of delivery (0=vaginal,1=cesarean)": {0, 1},
    "newborn sex (0=female,1=male)": {0, 1},
    "breastfeeding initiated": {0, 1},
    "referral to higher care": {0, 1},
}

for col, allowed in coded_domains.items():
    value = mock_llm_response[col]["value"]
    if value is not None:
        assert value in allowed, f"{col}: unexpected coded value {value}"

print("Validation passed.")


Validation passed.


## 6. Separate known, review, and missing fields

In [6]:
known_fields = {}
review_queue = []
missing_fields = []

for col in TARGET_COLUMNS:
    item = mock_llm_response[col]

    if item["status"] == "KNOWN":
        known_fields[col] = item["value"]

    elif item["status"] in {"NEEDS_REVIEW", "ILLEGIBLE"}:
        review_queue.append({
            "field": col,
            "proposed_value": item["value"],
            "evidence": item["evidence"],
            "reason": item["reason"],
            "question": item["question"]
        })

    elif item["status"] == "NOT_PROVIDED":
        missing_fields.append(col)

print("KNOWN:")
print(json.dumps(known_fields, indent=2))

print("\nREVIEW:")
print(json.dumps(review_queue, indent=2))

print("\nNOT PROVIDED:")
for field_name in missing_fields:
    print("-", field_name)


KNOWN:
{
  "id": 201,
  "age (years)": 28,
  "education level (0=none/primary,1=secondary,2=higher)": 1,
  "desired pregnancy": 1,
  "gravidity (number)": 2,
  "parity (number)": 1,
  "abortions (number)": 0,
  "living children (number)": 1,
  "previous cesarean": 0,
  "hemoglobin (g/dl)": 12.4,
  "first fasting glucose (mg/dl)": 86.0,
  "hiv test result": 0,
  "hepatitis c test result": 0
}

REVIEW:
[
  {
    "field": "bmi pregestational (kg/m2)",
    "proposed_value": 24.8,
    "evidence": "24.8",
    "reason": "The value is plausible for BMI, but the OCR fragment has no explicit BMI label.",
    "question": "I found an unlabeled value 24.8. Is this the pregestational BMI?"
  },
  {
    "field": "mean systolic bp (mmhg)",
    "proposed_value": 120,
    "evidence": "bp maybe 12O 8O",
    "reason": "OCR contains ambiguous O/0 characters.",
    "question": "I read the blood pressure as 120/80. Is that correct?"
  },
  {
    "field": "mean diastolic bp (mmhg)",
    "proposed_value": 80,


## 7. Simulate user review

Here the user:
- confirms blood pressure,
- confirms 24.8 is BMI,
- confirms 32 weeks is gestational age at enrollment.

No second LLM call is necessary for these simple confirmations.


In [7]:
confirmed = {
    key: dict(value)
    for key, value in mock_llm_response.items()
}

for col in [
    "bmi pregestational (kg/m2)",
    "mean systolic bp (mmhg)",
    "mean diastolic bp (mmhg)",
    "gestational age at enrollment (weeks)"
]:
    confirmed[col]["status"] = "KNOWN"

print("Remaining review fields:",
      sum(v["status"] in {"NEEDS_REVIEW", "ILLEGIBLE"} for v in confirmed.values()))


Remaining review fields: 0


## 8. Convert to the exact 31-column storage format

Only the final values are kept.

All temporary LLM metadata (`status`, `evidence`, `reason`, `question`) is discarded before storage.


In [8]:
import pandas as pd

final_record = {
    col: confirmed[col]["value"]
    for col in TARGET_COLUMNS
}

final_df = pd.DataFrame([final_record], columns=TARGET_COLUMNS)

print(final_df.to_string(index=False))


 id  age (years)  education level (0=none/primary,1=secondary,2=higher) consanguinity  desired pregnancy hypertension history diabetes mellitus  gravidity (number)  parity (number)  abortions (number)  living children (number)  previous cesarean  bmi pregestational (kg/m2)  mean systolic bp (mmhg)  mean diastolic bp (mmhg)  hemoglobin (g/dl)  first fasting glucose (mg/dl) proteinuria  hiv test result syphilis test result  hepatitis c test result  gestational age at enrollment (weeks) gestational dm gestational age at birth (weeks) preterm birth type of delivery (0=vaginal,1=cesarean) newborn sex (0=female,1=male) child birth weight (g) head circumference (cm) breastfeeding initiated referral to higher care
201           28                                                      1          None                  1                 None              None                   2                1                   0                         1                  0                        24.8           

## 9. Verify schema compatibility

In [9]:
assert list(final_df.columns) == TARGET_COLUMNS
assert final_df.shape == (1, 31)

print("Final shape:", final_df.shape)
print("Exact column order match:", list(final_df.columns) == TARGET_COLUMNS)


Final shape: (1, 31)
Exact column order match: True


## 10. Why this is better for real OCR

A weak implementation assumes:

```text
line 1 → field 1
line 2 → field 2
line 3 → field 3
```

This notebook instead assumes:

```text
whole OCR document
      ↓
find evidence anywhere
      ↓
align evidence to schema
```

That means the LLM can handle:

```text
g2 p1 a0
hiv neg
28
bp maybe 12O 8O
sec
```

without relying on input order.

### Important safety behavior

For an unlabeled fragment such as:

```text
24.8
```

the model should not automatically declare that it is BMI simply because the number looks plausible.

Instead:

```json
{
  "value": 24.8,
  "status": "NEEDS_REVIEW",
  "evidence": "24.8",
  "question": "I found an unlabeled value 24.8. Is this the pregestational BMI?"
}
```

The model performs **schema-aware interpretation**, but the user resolves uncertain mappings.


## 11. Final LLM-only architecture

```text
              NOISY OCR
                  ↓
        ┌───────────────────┐
        │     Local LLM     │
        │                   │
        │ Search all text   │
        │ Map to schema     │
        │ Normalize codes   │
        │ Preserve evidence │
        │ Flag ambiguity    │
        └─────────┬─────────┘
                  ↓
        temporary extraction JSON
                  ↓
        deterministic validator
             ┌────┴────┐
             ↓         ↓
           known     uncertain
             │         ↓
             │      ask user
             │         │
             └────┬────┘
                  ↓
        final 31-column record
```

The laptop can remain stateless: the temporary extraction object only needs to exist in memory for the current request.
